# 12. 희귀 유전자 열 제거 (H12)

## 배경
- 10 최적 설정(depth 3, colsample 0.3) → CV 0.4113 / LB 0.3206 (현재 최고)
- 11 핫스팟: CV +0.009였지만 LB −0.004 → 피처 추가로 오른 CV는 LB로 안 넘어감 (07 E와 같은 패턴)
- 반대로 과적합을 줄인 변경(09 모델 교체, 10 규제)은 LB로 넘어감
- 유전자 4,384개 중 변이가 극소수 환자에게만 있는 열은, 그 몇 명을 외우는 데 쓰일 가능성

## 가설
- H12: train에서 변이 환자가 k명 미만인 유전자 열을 빼면 과적합이 줄어 LB가 오른다
- H0: 트리는 쓸모없는 열을 알아서 안 쓰므로, 빼도 변화가 없다

## 검증 방법
- 모델: 10 최적 설정 고정, 피처: any + stats에서 희귀 유전자 열만 제거
- "k명 미만"은 여러 환자 통계 → Pipeline 안에서 fold 학습 부분으로만 셈 (Data Leakage 방지)
- k ∈ {3, 5, 10, 20} 비교 (k=1은 변이 0명 열 제거라 트리에는 의미 없음)
- 판단 기준: CV가 기준과 비슷하거나(std 이내) 높고 std가 작아지면 1개 제출. LB가 0.3206보다 오르면 H12 채택
  - 이번엔 피처를 줄이는 실험이라, CV가 조금 떨어져도 LB는 오를 수 있음 → CV 하락이 std 이내면 제출 후보로 둠

#### 준비
- 데이터, 피처(any + stats), `BalancedXGB`, `BEST`는 11과 동일
- `GENES`: 유전자 열 이름 목록 → 제거 대상은 이 열들만 (stats 9열은 항상 유지)

In [1]:
import sys; sys.path.append('..')
import time, numpy as np, pandas as pd, xgboost as xgb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)
X      = pd.concat([F_tr['any'], F_tr['stats']], axis=1)
X_test = pd.concat([F_te['any'], F_te['stats']], axis=1)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
GENES = list(F_tr['any'].columns)

#### 유전자별 변이 환자 수 분포 (train만)
- k를 정하기 전에, k명 미만인 유전자가 몇 개인지 확인
- train 라벨이나 test는 쓰지 않는 단순 집계 (EDA)

In [2]:
n_per_gene = F_tr['any'].sum()
print(f'유전자 {len(n_per_gene):,}개 | 변이 환자 수 중앙값 {n_per_gene.median():.0f}명')
for k in [1, 3, 5, 10, 20]:
    print(f'  {k}명 미만: {(n_per_gene < k).sum():,}개 → 남는 유전자 {(n_per_gene >= k).sum():,}개')

유전자 4,384개 | 변이 환자 수 중앙값 36명
  1명 미만: 154개 → 남는 유전자 4,230개
  3명 미만: 169개 → 남는 유전자 4,215개
  5명 미만: 213개 → 남는 유전자 4,171개
  10명 미만: 415개 → 남는 유전자 3,969개
  20명 미만: 1,065개 → 남는 유전자 3,319개


#### fold 안에서 희귀 유전자를 고르는 변환기
- `RareGeneFilter`: `fit`에서 fold 학습 부분의 유전자별 변이 환자 수를 세고, k명 미만인 열을 버릴 목록으로 기억 → `transform`에서 제거
- 11의 `FrequentTokens`와 같은 구조 → 검증 fold 환자는 목록 선정에 관여하지 않음
- `n_kept_`: 실제로 남은 유전자 수 기록

In [3]:
class RareGeneFilter(BaseEstimator, TransformerMixin):
    def __init__(self, gene_cols, min_count=5):
        self.gene_cols, self.min_count = gene_cols, min_count

    def fit(self, X, y=None):
        n = X[self.gene_cols].sum()
        self.drop_ = n.index[n < self.min_count].tolist()
        self.n_kept_ = len(self.gene_cols) - len(self.drop_)
        return self

    def transform(self, X):
        return X.drop(columns=self.drop_)

def make_rare(k):
    return lambda: make_pipeline(RareGeneFilter(GENES, k), BalancedXGB(**BEST))

#### 기준 vs 희귀 유전자 제거 (k 4가지)
- 첫 줄은 10 최적 설정 재현 (0.4113 확인)
- k가 클수록 많이 제거 → 점수와 std가 어떻게 변하는지 확인

In [5]:
rows = []
t = time.time()
r = run_cv(lambda: BalancedXGB(**BEST), X, y, verbose=False)
rows.append(dict(name='10_best (기준)', n_genes=len(GENES), cv=round(r['cv_mean'], 4),
                 std=round(r['cv_std'], 4), sec=round(time.time() - t)))
print(rows[-1])

for k in [3, 5, 10, 20]:
    t = time.time()
    r = run_cv(make_rare(k), X, y, verbose=False)
    n_genes = int(np.mean([m.named_steps['raregenefilter'].n_kept_ for m in r['models']]))
    rows.append(dict(name=f'rare_k{k}', n_genes=n_genes, cv=round(r['cv_mean'], 4),
                     std=round(r['cv_std'], 4), sec=round(time.time() - t)))
    print(rows[-1])

res12 = pd.DataFrame(rows).sort_values('cv', ascending=False)
res12

{'name': '10_best (기준)', 'n_genes': 4384, 'cv': 0.4113, 'std': 0.0024, 'sec': 151}
{'name': 'rare_k3', 'n_genes': 4200, 'cv': 0.4135, 'std': 0.0041, 'sec': 136}
{'name': 'rare_k5', 'n_genes': 4134, 'cv': 0.4118, 'std': 0.0041, 'sec': 132}
{'name': 'rare_k10', 'n_genes': 3826, 'cv': 0.4119, 'std': 0.0102, 'sec': 135}
{'name': 'rare_k20', 'n_genes': 2981, 'cv': 0.4115, 'std': 0.004, 'sec': 107}


,name,n_genes,cv,std,sec
1,rare_k3,4200,0.4135,0.0041,136
3,rare_k10,3826,0.4119,0.0102,135
2,rare_k5,4134,0.4118,0.0041,132
4,rare_k20,2981,0.4115,0.0040,107
0,10_best (기준),4384,0.4113,0.0024,151


#### 제출 파일 생성
- 셀 5 결과로 고른 k로 제출
- 선택 기준: 기준과 CV 차이가 std 이내라면, 유전자를 더 많이 뺀(단순한) 쪽을 우선

In [6]:
K = 20
r = run_cv(make_rare(K), X, y, X_test)
save_submission(submission, r['test_proba'], le, name=f'xgb_d3_cs0.3_rare{K}', cv_score=r['cv_mean'])

[fold 0] Macro F1 = 0.4170
[fold 1] Macro F1 = 0.4090
[fold 2] Macro F1 = 0.4154
[fold 3] Macro F1 = 0.4066
[fold 4] Macro F1 = 0.4095
CV Macro F1 = 0.4115 ± 0.0040 (OOF 0.4121)
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1110_xgb_d3_cs0.3_rare20_0.412.csv


WindowsPath('C:/Users/USER/Desktop/hackathon/submissions/1110_xgb_d3_cs0.3_rare20_0.412.csv')

**k20을 선택한 이유**
- CV는 k3~k20과 기준이 모두 0.4113~0.4135로 차이가 std(약 0.004) 이내 → CV로는 우열 없음
- 계획서 기준대로, 유전자를 가장 많이 뺀(1,403개, 32%) k20 선택 → H12 효과를 가장 분명하게 확인 가능
- k10은 std 0.0102로 불안정해 제외
- 학습 시간 151초 → 107초로 감소

**제출**: k20 → LB 0.3051 (10 대비 −0.0155)

## 결론
- **H12 기각**: CV는 기준과 같았지만(0.4115 vs 0.4113) LB는 오차 범위를 넘게 하락
- train에서 드문 유전자도 test 예측에는 필요했음
  - 추정 1: 소수 아형(DLBC 38명, ACC 72명 등)의 특징 유전자는 애초에 변이 환자 수가 적을 수밖에 없음 → 제거 시 Macro F1 손실
  - 추정 2: test는 변이 비율이 높아 보임 → train에서 드물던 유전자가 test에서 더 자주 등장했을 가능성
- **교훈 갱신**: LB로 통한 건 "모델이 덜 외우게 하기"(09, 10)였고, "정보 버리기"(12)는 손해
- CV가 같아도 LB는 크게 다를 수 있음 → 10 vs 12-k20은 13 강건성 점검 CV의 검증용 쌍으로 사용
- 최고 LB는 10 (0.3206) 유지